# 🏛️ LegisSum — Colab Training (Mistral-7B + QLoRA)

**Runtime → Change runtime type → T4 GPU** before running!

This notebook:
1. Installs dependencies
2. Uploads your preprocessed data from Mac
3. Fine-tunes Mistral-7B with QLoRA (~3–4 hrs on free T4)
4. Downloads the LoRA adapter back to your Mac

---

In [ ]:
# ── Cell 1: Confirm GPU ──────────────────────────────────────────────────────
import torch
print('GPU available:', torch.cuda.is_available())
print('Device name  :', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None')
# Should print: Tesla T4

In [ ]:
# ── Cell 2: Install dependencies ─────────────────────────────────────────────
!pip install -q -U \
    transformers>=4.40.0 \
    datasets>=2.18.0 \
    peft>=0.10.0 \
    trl>=0.8.6 \
    bitsandbytes>=0.44.0 \
    accelerate>=0.29.0 \
    rouge-score


In [ ]:
# ── Cell 3: Upload processed data from your Mac ──────────────────────────────
# SKIP THIS — use Cell 3b below (auto-download) instead
# If you already have train.jsonl locally, you can upload manually:
#
# from google.colab import files
# import os
# os.makedirs('data/processed', exist_ok=True)
# uploaded = files.upload()
# for fname in uploaded:
#     os.rename(fname, f'data/processed/{fname}')


In [ ]:
# ── Cell 3b: Auto-download & preprocess BillSum directly in Colab ──────────
# Recommended: no file upload needed

from datasets import load_dataset
import json, os

SYSTEM = (
    'You are a legislative assistant helping everyday citizens understand U.S. law. '
    'Read the following Congressional bill and write a clear, concise summary. '
    'Focus on: what the bill does, who it affects, and its key provisions. '
    'Use plain English — no legal jargon.'
)

ds = load_dataset('FiscalNote/billsum')
os.makedirs('data/processed', exist_ok=True)

def fmt(row):
    inp = f"{row['title']}\n\n{row['text']}"[:14000]
    return {
        'input': inp,
        'output': row['summary'],
        'text': f"### Instruction:\n{SYSTEM}\n\n### Bill:\n{inp}\n\n### Summary:\n{row['summary']}"
    }

train_rows = []
with open('data/processed/train.jsonl', 'w') as f:
    for row in ds['train']:
        words_text = len(row['text'].split())
        words_summary = len(row['summary'].split())
        if words_text > 200 and words_summary > 20:
            record = fmt(row)
            f.write(json.dumps(record) + '\n')
            train_rows.append(record)

print(f'✅ Saved {len(train_rows)} training samples to data/processed/train.jsonl')


In [ ]:
# ── Cell 4: HuggingFace login (needed for Mistral gated model) ───────────────
from huggingface_hub import login
login()  # paste your HF token when prompted
# Get token at: https://huggingface.co/settings/tokens
# Also accept Mistral license at: https://huggingface.co/mistralai/Mistral-7B-Instruct-v0.3

In [ ]:
# ── Cell 5: Fine-tune Mistral-7B with QLoRA ───────────────────────────────────
import json, torch
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig

BASE_MODEL  = 'mistralai/Mistral-7B-Instruct-v0.3'
MODEL_OUT   = './billsum-lora'

# Set your HuggingFace username to auto-push checkpoints
# This saves your adapter to HF Hub so Colab disconnects won't lose progress
import os
HF_USERNAME = ''  # ← FILL IN your HF username (e.g. 'yashb')
HUB_MODEL_ID = f'{HF_USERNAME}/legissum-mistral-lora' if HF_USERNAME else None

# Load data
with open('data/processed/train.jsonl') as f:
    data = [json.loads(l) for l in f]
train_ds = Dataset.from_list(data)
print(f'Training on {len(train_ds)} samples')

# Tokenizer
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

# 4-bit QLoRA
bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)
model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, quantization_config=bnb, device_map='auto')
model = prepare_model_for_kbit_training(model)

lora_cfg = LoraConfig(
    r=16, lora_alpha=32,
    target_modules=['q_proj','v_proj','k_proj','o_proj'],
    lora_dropout=0.05, bias='none', task_type='CAUSAL_LM'
)
model = get_peft_model(model, lora_cfg)
model.print_trainable_parameters()

trainer = SFTTrainer(
    model=model,
    train_dataset=train_ds,
    tokenizer=tokenizer,
    args=SFTConfig(
        output_dir=MODEL_OUT,
        num_train_epochs=2,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        learning_rate=2e-4,
        fp16=True,
        logging_steps=50,
        save_steps=500,
        dataset_text_field='text',
        max_seq_length=1024,
        report_to='none',
        push_to_hub=bool(HUB_MODEL_ID),
        hub_model_id=HUB_MODEL_ID,
        save_strategy='steps',
        warmup_ratio=0.05,
    ),
)

print('🚀 Training started...')
trainer.train()
trainer.save_model(MODEL_OUT)
tokenizer.save_pretrained(MODEL_OUT)
print('✅ Model saved to', MODEL_OUT)

In [ ]:
# ── Cell 6: Quick Before/After spot check ────────────────────────────────────
from peft import PeftModel

def summarize(model, tokenizer, text, max_new=250):
    SYSTEM = (
        "You are a legislative assistant helping everyday citizens understand U.S. law. "
        "Read the following Congressional bill and write a clear, concise summary."
    )
    prompt = f"### Instruction:\n{SYSTEM}\n\n### Bill:\n{text[:2000]}\n\n### Summary:\n"
    inputs = tokenizer(prompt, return_tensors='pt', truncation=True).to('cuda')
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new, temperature=0.3,
                             do_sample=True, pad_token_id=tokenizer.eos_token_id)
    decoded = tokenizer.decode(out[0], skip_special_tokens=True)
    return decoded.split('### Summary:')[-1].strip()

# Load test sample
test_sample = data[500]  # pick any sample
bill_text   = test_sample['input'] if 'input' in test_sample else test_sample['text']

# Base model (BEFORE)
base = AutoModelForCausalLM.from_pretrained(BASE_MODEL, quantization_config=bnb, device_map='auto')
print('❌ BEFORE (base model):')
print(summarize(base, tokenizer, bill_text))
del base

# Fine-tuned (AFTER)
ft_base = AutoModelForCausalLM.from_pretrained(BASE_MODEL, quantization_config=bnb, device_map='auto')
ft_model = PeftModel.from_pretrained(ft_base, MODEL_OUT)
print('\n✅ AFTER (fine-tuned):')
print(summarize(ft_model, tokenizer, bill_text))

print('\n🏛️ Ground truth:')
print(test_sample.get('output', '(not available)'))

In [ ]:
# ── Cell 7: Download adapter back to your Mac ────────────────────────────────
import shutil
shutil.make_archive('billsum-lora', 'zip', '.', 'billsum-lora')

from google.colab import files
files.download('billsum-lora.zip')
print('Downloaded! Unzip into legissum/models/billsum-lora/ on your Mac.')

# ── Also push to HuggingFace Hub (run this too!) ─────────────────────────────
# This lets the demo app load the model without local files
if HUB_MODEL_ID:
    ft_model.push_to_hub(HUB_MODEL_ID)
    tokenizer.push_to_hub(HUB_MODEL_ID)
    print(f'✅ Pushed to HF Hub: {HUB_MODEL_ID}')
    print(f'   Set HF_MODEL_ID={HUB_MODEL_ID} in your .env to load it in app.py')
else:
    print('Tip: set HF_USERNAME in Cell 5 to also push to HF Hub next time')
